# connections-rl — multi-seed behavioral eval (one session, both scales)

Per-seed solve / invalid / groups-correct on the held-out test split for GRPO
seeds 0, 1, 2 at both scales, plus across-seed variance. Completes the
multi-seed evidence alongside the weight-space convergence result.

Settings → Accelerator → **GPU T4 x2**, Internet → On. Save & Run All, walk away.
~2-3h: 7B first (tp=2, eager), then the 1.5B arms on a single GPU.

Results auto-push to the Hub dataset repo, so nothing depends on the session
surviving or on remembering to download.

In [ ]:
# Cell 1 — setup: repo, data, all six adapters (2 SFT + 3 GRPO seeds per scale)
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e . openai vllm
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

from huggingface_hub import snapshot_download
REPOS = {
    # 7B
    'sft-7b':        'connections-rl-sft-7b',
    'grpo-7b-s0':    'connections-rl-grpo-7b',
    'grpo-7b-s1':    'connections-rl-grpo-7b-seed1',
    'grpo-7b-s2':    'connections-rl-grpo-7b-seed2',
    # 1.5B
    'sft-1.5b':      'connections-rl-sft',
    'grpo-1.5b-s0':  'connections-rl-grpo',
    'grpo-1.5b-s1':  'connections-rl-grpo-1.5b-seed1',
    'grpo-1.5b-s2':  'connections-rl-grpo-1.5b-seed2',
}
for local, repo in REPOS.items():
    snapshot_download(f'{HF_USER}/{repo}', local_dir=f'adapters/{local}',
                      token=os.environ['HF_TOKEN'], allow_patterns=['adapter*', '*.json', '*.jinja'])
    assert os.path.exists(f'adapters/{local}/adapter_config.json'), local
print('all 8 adapters ready')

In [ ]:
# Cell 2 — vLLM 7B (tp=2, fp16, eager) with SFT + all three GRPO seeds
import subprocess, time, urllib.request

A = '/kaggle/working/connections-rl/adapters'

def serve(cmd, log, tries=150):
    p = subprocess.Popen(cmd, shell=True, stdout=open(log, 'w'), stderr=subprocess.STDOUT)
    for _ in range(tries):
        try:
            urllib.request.urlopen('http://localhost:8000/health'); print('vLLM ready'); return p
        except Exception: time.sleep(10)
    raise RuntimeError(f'vLLM failed - check {log}')

proc = serve(
    f'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
    f'--enable-lora --enforce-eager --max-loras 2 '
    f'--lora-modules connections-rl-sft-7b={A}/sft-7b '
    f'connections-rl-grpo-7b={A}/grpo-7b-s0 '
    f'connections-rl-grpo-7b-seed1={A}/grpo-7b-s1 '
    f'connections-rl-grpo-7b-seed2={A}/grpo-7b-s2 '
    f'--max-model-len 2048 --gpu-memory-utilization 0.85',
    '/kaggle/working/vllm-7b.log')

In [ ]:
# Cell 3 — 7B: greedy eval of every seed (same protocol as the published table)
!python -m connections_rl.eval.run --config configs/eval/seeds-7b.yaml

In [ ]:
# Cell 4 — swap to 1.5B (single GPU)
proc.terminate(); time.sleep(20)
!pkill -f 'vllm serve' || true
time.sleep(25)
proc2 = serve(
    f'vllm serve Qwen/Qwen2.5-1.5B-Instruct --dtype half --enable-lora --enforce-eager '
    f'--max-loras 2 '
    f'--lora-modules connections-rl-sft={A}/sft-1.5b '
    f'connections-rl-grpo={A}/grpo-1.5b-s0 '
    f'connections-rl-grpo-1.5b-seed1={A}/grpo-1.5b-s1 '
    f'connections-rl-grpo-1.5b-seed2={A}/grpo-1.5b-s2 '
    f'--max-model-len 2048 --gpu-memory-utilization 0.85',
    '/kaggle/working/vllm-1.5b.log')

In [ ]:
# Cell 5 — 1.5B: greedy eval of every seed
!python -m connections_rl.eval.run --config configs/eval/seeds-1.5b.yaml

In [ ]:
# Cell 6 — across-seed variance summary (the paper table)
import json, statistics as st
from pathlib import Path

rows = []
for scale, out in [('7B', 'results-seeds-7b'), ('1.5B', 'results-seeds-1.5b')]:
    for arm in ['sft', 'grpo-seed0', 'grpo-seed1', 'grpo-seed2']:
        f = Path(out) / arm / 'metrics.json'
        if not f.exists():
            print('missing', f); continue
        o = json.loads(f.read_text())['summary']['OVERALL']
        rows.append({'scale': scale, 'arm': arm, 'solve': o['solve_rate'][0],
                     'groups': o['groups_correct'][0], 'invalid': o['invalid_rate'][0],
                     'reward': o['reward'][0]})

print(f"{'scale':<6}{'arm':<12}{'solve':>8}{'groups':>9}{'invalid':>9}{'reward':>9}")
for r in rows:
    print(f"{r['scale']:<6}{r['arm']:<12}{r['solve']:>8.3f}{r['groups']:>9.3f}"
          f"{r['invalid']:>9.3f}{r['reward']:>9.3f}")

print('\n=== across-seed spread (GRPO seeds 0/1/2) ===')
summary = {}
for scale in ('7B', '1.5B'):
    g = [r for r in rows if r['scale'] == scale and r['arm'].startswith('grpo')]
    if len(g) < 2: continue
    summary[scale] = {}
    for k in ('solve', 'groups', 'invalid', 'reward'):
        vals = [r[k] for r in g]
        summary[scale][k] = {'mean': st.mean(vals), 'sd': st.stdev(vals),
                             'min': min(vals), 'max': max(vals), 'n_seeds': len(vals)}
        print(f"{scale:<6}{k:<9} mean={st.mean(vals):.3f}  sd={st.stdev(vals):.3f}  "
              f"range=[{min(vals):.3f}, {max(vals):.3f}]")

Path('results-seeds').mkdir(exist_ok=True)
Path('results-seeds/seed_summary.json').write_text(
    json.dumps({'per_arm': rows, 'across_seed': summary}, indent=1))
print('\nwrote results-seeds/seed_summary.json')

In [ ]:
# Cell 7 — durable copy to the Hub + local link
from huggingface_hub import HfApi
from IPython.display import FileLink, display
api = HfApi()
repo = f'{HF_USER}/connections-rl-results'
api.create_repo(repo, repo_type='dataset', exist_ok=True)
for folder in ['results-seeds-7b', 'results-seeds-1.5b', 'results-seeds']:
    if os.path.isdir(folder):
        api.upload_folder(folder_path=folder, repo_id=repo, repo_type='dataset',
                          path_in_repo=folder)
        print('pushed', folder)
print(f'-> huggingface.co/datasets/{repo}')
!zip -qr /kaggle/working/results-seeds.zip results-seeds results-seeds-7b results-seeds-1.5b
display(FileLink('/kaggle/working/results-seeds.zip'))